In [1]:
import pandas as pd

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load the merged file
input_file = Path('../active_hour/active_hour_results/active_hour.csv')
output_dir = Path('./streak_results')

df = pd.read_csv(input_file)

In [3]:
# Convert date columns
df['assign_created_at'] = pd.to_datetime(df['assign_created_at'])
df['date'] = pd.to_datetime(df['date']).dt.date
df['week_start_date'] = pd.to_datetime(df['week_start_date']).dt.date

# Define target day sets
target1_days = {'Mon', 'Tue'}
target2_days = {'Wed', 'Thu', 'Fri'}
target3_days = {'Sat', 'Sun'}

# Identify which target period each order belongs to
df['is_target1_day'] = df['day_of_week_short'].isin(target1_days)
df['is_target2_day'] = df['day_of_week_short'].isin(target2_days)
df['is_target3_day'] = df['day_of_week_short'].isin(target3_days)

# Count only if order is accepted and in the target period
df['is_accepted_target1'] = df['is_target1_day'] & df['is_driver_accept']
df['is_accepted_target2'] = df['is_target2_day'] & df['is_driver_accept']
df['is_accepted_target3'] = df['is_target3_day'] & df['is_driver_accept']

# Calculate cumulative accepted order counts for each target period
df['orders_target1_cumulative'] = df.groupby(['driver_id', 'week_start_date'])['is_accepted_target1'].cumsum()
df['orders_target2_cumulative'] = df.groupby(['driver_id', 'week_start_date'])['is_accepted_target2'].cumsum()
df['orders_target3_cumulative'] = df.groupby(['driver_id', 'week_start_date'])['is_accepted_target3'].cumsum()


In [4]:
# Get unique dates per driver per week for target completion tracking
daily_summary = df.groupby(['driver_id', 'week_start_date', 'date']).agg({
    'day_of_week_short': 'first',
    'year': 'first',
    'week': 'first'
}).reset_index()

# Sort by driver, week_start_date, and date
daily_summary = daily_summary.sort_values(['driver_id', 'week_start_date', 'date'])

In [5]:
def get_target_days_info(group):
    """
    For each day in the week, track which target day sets have been FULLY completed so far.
    Target 1: Mon AND Tue (both required)
    Target 2: Wed AND Thu AND Fri (all three required)
    Target 3: Sat AND Sun (both required)
    """
    target1_complete = []
    target2_complete = []
    target3_complete = []
    
    days_worked_so_far = set()
    
    for day in group['day_of_week_short'].values:
        days_worked_so_far.add(day)
        
        # Check if ALL target days have been worked
        target1_complete.append(target1_days.issubset(days_worked_so_far))
        target2_complete.append(target2_days.issubset(days_worked_so_far))
        target3_complete.append(target3_days.issubset(days_worked_so_far))
    
    return pd.DataFrame({
        'driver_id': group['driver_id'].values,
        'week_start_date': group['week_start_date'].values,
        'date': group['date'].values,
        'worked_target1_days': target1_complete,  # Worked BOTH Mon AND Tue
        'worked_target2_days': target2_complete,  # Worked ALL Wed, Thu, AND Fri
        'worked_target3_days': target3_complete   # Worked BOTH Sat AND Sun
    })

# Process each (driver_id, week_start_date) group
target_summary = daily_summary.groupby(['driver_id', 'week_start_date'], group_keys=False).apply(
    get_target_days_info
).reset_index(drop=True)

/var/folders/s_/0td42yfs6wjd9wdpc5nr90400000gn/T/ipykernel_52906/2402740879.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  target_summary = daily_summary.groupby(['driver_id', 'week_start_date'], group_keys=False).apply(


In [6]:
# Merge back to dataset
# Convert date types to match for merging
target_summary['date'] = pd.to_datetime(target_summary['date']).dt.date
target_summary['week_start_date'] = pd.to_datetime(target_summary['week_start_date']).dt.date

df = df.merge(
    target_summary[['driver_id', 'week_start_date', 'date', 
                    'worked_target1_days', 'worked_target2_days', 'worked_target3_days']],
    on=['driver_id', 'week_start_date', 'date'],
    how='left'
)

#save results
output_path = output_dir / 'streak_results.csv'
df.to_csv(output_path, index=False)
print(f"\nSaved updated dataset: {output_path}")
print(f"Total records: {len(df):,}")


Saved updated dataset: streak_results/streak_results.csv
Total records: 6,164,585


In [7]:
streak = pd.read_csv('./streak_results/streak_results.csv')

In [8]:
streak.head(10)

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,is_target3_day,is_accepted_target1,is_accepted_target2,is_accepted_target3,orders_target1_cumulative,orders_target2_cumulative,orders_target3_cumulative,worked_target1_days,worked_target2_days,worked_target3_days
0,0,SURIN,LMD00BZBZ,2025-07-10 10:27:54.432,ก๋วยเตี๋ยวถนอมมิตร2 (Thanommit Noodles 2),Noodles,LMF-250710-408959062,True,13.50,3.0,...,False,False,True,False,0,1,0,False,False,False
1,1,SURIN,LMD00BZBZ,2025-07-10 10:32:29.887,ตำซี๊ด&อาหารตามสั่ง (Tum Zeed & Made-to-Order ...,Dessert,LMF-250710-161870697,True,17.80,3.0,...,False,False,True,False,0,2,0,False,False,False
2,2,SURIN,LMD00BZBZ,2025-07-10 10:47:04.631,เฉิงกง ติ่มซำ (Chenggong Dim Sum) -,Dim Sum,LMF-250710-166811972,True,12.66,3.0,...,False,False,True,False,0,3,0,False,False,False
3,3,SURIN,LMD00BZBZ,2025-07-10 10:56:22.791,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-400808907,True,12.37,3.0,...,False,False,True,False,0,4,0,False,False,False
4,4,SURIN,LMD00BZBZ,2025-07-10 11:09:04.244,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-858627724,True,14.30,3.0,...,False,False,True,False,0,5,0,False,False,False
5,5,SURIN,LMD00BZBZ,2025-07-10 11:11:38.237,Café Amazon - SD3192 (คาเฟ่ อเมซอน - SD3192) ส...,Café/Coffee Shop,LMF-250710-321972404,True,12.37,3.0,...,False,False,True,False,0,6,0,False,False,False
6,6,SURIN,LMD00BZBZ,2025-07-10 11:23:47.869,Café Amazon - DD1655 (คาเฟ่ อเมซอน - DD1655) ป...,Café/Coffee Shop,LMF-250710-771935451,True,14.30,3.0,...,False,False,True,False,0,7,0,False,False,False
7,7,SURIN,LMD00BZBZ,2025-07-10 11:33:30.396,PARLOR,Cafe,LMF-250710-015980523,True,11.40,3.0,...,False,False,True,False,0,8,0,False,False,False
8,8,SURIN,LMD00BZBZ,2025-07-10 11:40:19.467,PunThai Coffee (กาแฟพันธุ์ไทย) ถนนหลักเมือง สุ...,Café/Coffee Shop,LMF-250710-466168679,True,11.40,3.0,...,False,False,True,False,0,9,0,False,False,False
9,9,SURIN,LMD00BZBZ,2025-07-10 11:49:49.241,ลุงแหลมอาหารตามสั่งหน้าเทคนิค (Uncle Laem's Ma...,À La Carte,LMF-250710-422617314,True,11.40,3.0,...,False,False,True,False,0,10,0,False,False,False


In [9]:
LMD00BZBZ = streak[streak['driver_id'] == 'LMD00BZBZ']

In [10]:
LMD00BZBZ.columns

Index(['Unnamed: 0', 'region', 'driver_id', 'assign_created_at',
       'restaurant_name', 'restaurant_category', 'order_id',
       'is_driver_accept', 'wage', 'on_top_fare', 'basket_size',
       'drop_off_distance', 'distance_rider_to_restaurant',
       'placed_to_match_second', 'rider_matched_to_arrived_restaurant_second',
       'rider_waiting_time_in_restaurant_second',
       'delivery_time_to_user_second', 'total_user_waiting_time_second',
       'order_status', 'date', 'region::multi-filter', 'avg_rain',
       'Rain_Level', 'Rider Group', 'Start Date', 'End Date', 'Incentive',
       'daily_order_count', 'time_per_order_sec', 'time_worked_sec', 'hour',
       'daily_cumulative_time_sec', 'time_worked_in_window_sec',
       'window_cumulative_sec', 'daily_cumulative_time_hr',
       'window_cumulative_hr', 'Unnamed: 0.1', 'day_of_week',
       'day_of_week_short', 'week_start_date', 'year', 'week',
       'days_worked_count', 'days_worked_list', 'has_required_days',
       'i

In [11]:
LMD00BZBZ[(LMD00BZBZ['week'] == 30)].tail(30)

,Unnamed: 0,region,driver_id,assign_created_at,restaurant_name,restaurant_category,order_id,is_driver_accept,wage,on_top_fare,...,is_target3_day,is_accepted_target1,is_accepted_target2,is_accepted_target3,orders_target1_cumulative,orders_target2_cumulative,orders_target3_cumulative,worked_target1_days,worked_target2_days,worked_target3_days
441,12,SURIN,LMD00BZBZ,2025-07-26 09:43:18.612,PunThai Coffee (กาแฟพันธุ์ไทย) ถนนหลักเมือง สุ...,Café/Coffee Shop,LMF-250726-831631157,True,12.06,3.0,...,True,False,False,True,81,108,13,True,True,False
442,13,SURIN,LMD00BZBZ,2025-07-26 10:14:36.349,69 Tea House สุรินทร์,Café/Coffee Shop,LMF-250726-567266990,True,12.20,3.0,...,True,False,False,True,81,108,14,True,True,False
443,14,SURIN,LMD00BZBZ,2025-07-26 10:22:10.522,KFC (เคเอฟซี) สุรินทร์พลาซ่า,Fastfood,LMF-250726-167967059,True,12.20,3.0,...,True,False,False,True,81,108,15,True,True,False
444,15,SURIN,LMD00BZBZ,2025-07-26 10:29:18.294,ภูมิใจเสนอ ก๋วยเตี๋ยวลูกชิ้นปลา&ข้าวมันไก่,Noodles,LMF-250726-726695404,True,12.20,3.0,...,True,False,False,True,81,108,16,True,True,False
445,16,SURIN,LMD00BZBZ,2025-07-26 10:43:36.250,ข้าวมันไก่ KCP (Khaw man ki KCP),Thai,LMF-250726-605605517,True,13.80,3.0,...,True,False,False,True,81,108,17,True,True,False
446,17,SURIN,LMD00BZBZ,2025-07-26 10:51:02.617,ข้าวมันไก่ KCP (Khaw man ki KCP),Thai,LMF-250726-209670462,True,21.60,3.0,...,True,False,False,True,81,108,18,True,True,False
447,18,SURIN,LMD00BZBZ,2025-07-26 10:59:12.843,ครัวตะเกียง (Lantern Kitchen) ซอยสระโบราณ1,À La Carte,LMF-250726-848001422,True,13.80,3.0,...,True,False,False,True,81,108,19,True,True,False
448,19,SURIN,LMD00BZBZ,2025-07-26 11:22:00.745,ครัวคุณอ๋อย กะเพราโบราณ (Kroaw khung oil) ซอยเ...,Delivery Only,LMF-250726-501682846,True,13.70,3.0,...,True,False,False,True,81,108,20,True,True,False
449,20,SURIN,LMD00BZBZ,2025-07-26 11:24:13.759,เจ๊ติ๊กกาแฟโบราณ (Jae Tik Ancient Coffee),Café/Coffee Shop,LMF-250726-763722699,True,13.70,3.0,...,True,False,False,True,81,108,21,True,True,False
450,21,SURIN,LMD00BZBZ,2025-07-26 11:32:34.777,Hop Chafe,Bubble Milk Tea,LMF-250726-851748536,True,25.40,3.0,...,True,False,False,True,81,108,22,True,True,False
